Add logic for tiebreak scorekeeping, and indicators for critical points. (game, set and match point)

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1]
DATA_PROCESSED = ROOT / "data_processed"

in_path = DATA_PROCESSED / "02_serve_state_features.parquet"
df = pd.read_parquet(in_path)

Find tiebreaks - GameNo = 13, add indicators for points played in a tiebreak

In [3]:

df["GameNo"] = pd.to_numeric(df["GameNo"], errors="coerce").astype("Int64")
df["tiebreak"] = (df["GameNo"] == 13).astype("Int64")

print("Rows:", len(df))
print("Tiebreak points:", int(df["tiebreak"].sum()))

Rows: 48211
Tiebreak points: 1175


In [4]:
df['p1_tb_points'] = 0
df['p2_tb_points'] = 0
df['tb_current_score'] = ""

mask_tb = df['tiebreak'] == 1
tb = df[mask_tb].copy()

p1_tb = (tb['PointWinner'] == 1).groupby([tb['match_id'], tb['SetNo']]).cumsum().astype(int)
p2_tb = (tb['PointWinner'] == 2).groupby([tb['match_id'], tb['SetNo']]).cumsum().astype(int)

df.loc[mask_tb, 'p1_tb_points'] = p1_tb
df.loc[mask_tb, 'p2_tb_points'] = p2_tb


tiebreak scoreboard for debugging if needed.

In [11]:
df.loc[mask_tb, 'tb_current_score'] = (
    df.loc[mask_tb, 'p1_tb_points'].astype(int).astype(str)
    + '-' +
    df.loc[mask_tb, 'p2_tb_points'].astype(int).astype(str)
)

In [12]:
score_cols = [
    'p1_pts_game_before', 'p2_pts_game_before',
    'p1_pts_game_after',  'p2_pts_game_after'
]

Zero out non-tiebreak points.

In [13]:
df.loc[mask_tb, score_cols] = 0
df.loc[mask_tb, 'game_score_str'] = '0-0'

Validate tiebreak scorekeeping logic.

In [14]:
tb_set = (
    df.loc[df["tiebreak"] == 1, ["match_id", "SetNo"]]
    .drop_duplicates()
    .iloc[0]
)

match_id = tb_set["match_id"]
set_no = tb_set["SetNo"]

print("Example tiebreak set:", match_id, "Set", set_no)


Example tiebreak set: 2011-ausopen-2503 Set 2


In [15]:
set_df = (
    df[(df["match_id"] == match_id) & (df["SetNo"] == set_no)]
    .sort_values("PointNumber")
)

set_df[[
    "match_id",
    "SetNo",
    "GameNo",
    "PointNumber",
    "PointServer",
    "PointWinner",
    "p1_tb_points",
    "p2_tb_points",
    "tb_current_score",
    "p1_pts_game_before",
    "p2_pts_game_before",
    "game_score_str",
    "tiebreak"
]]


,match_id,SetNo,GameNo,PointNumber,PointServer,PointWinner,p1_tb_points,p2_tb_points,tb_current_score,p1_pts_game_before,p2_pts_game_before,game_score_str,tiebreak
343,2011-ausopen-2503,2,1,51,2,2,0,0,,0,0,0-0,0
344,2011-ausopen-2503,2,1,52,2,1,0,0,,0,1,0-15,0
345,2011-ausopen-2503,2,1,53,2,1,0,0,,1,1,15-15,0
346,2011-ausopen-2503,2,1,54,2,1,0,0,,2,1,30-15,0
347,2011-ausopen-2503,2,1,55,2,2,0,0,,3,1,40-15,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
439,2011-ausopen-2503,2,13,147,1,1,4,3,4-3,0,0,0-0,1
440,2011-ausopen-2503,2,13,148,2,2,4,4,4-4,0,0,0-0,1
441,2011-ausopen-2503,2,13,149,2,2,4,5,4-5,0,0,0-0,1
442,2011-ausopen-2503,2,13,150,1,2,4,6,4-6,0,0,0-0,1


Detect break points (non-serving player has a chance to win current game).

In [16]:
import numpy as np

df['server']   = np.where(df['is_p1_server'] == 1, 1, 2)
df['receiver'] = 3 - df['server']

p1_before = df['p1_pts_game_before']
p2_before = df['p2_pts_game_before']

p1_after_if_receiver_wins = p1_before + (df['receiver'] == 1).astype(int)
p2_after_if_receiver_wins = p2_before + (df['receiver'] == 2).astype(int)

receiver_wins_game_if_wins = (
    ((df['receiver'] == 1) &
     (p1_after_if_receiver_wins >= 4) &
     ((p1_after_if_receiver_wins - p2_before) >= 2))
    |
    ((df['receiver'] == 2) &
     (p2_after_if_receiver_wins >= 4) &
     ((p2_after_if_receiver_wins - p1_before) >= 2))
)

df['break_point_p1'] = 0
df['break_point_p2'] = 0

df.loc[(df['tiebreak'] == 0) & receiver_wins_game_if_wins & (df['receiver'] == 1), 'break_point_p1'] = 1
df.loc[(df['tiebreak'] == 0) & receiver_wins_game_if_wins & (df['receiver'] == 2), 'break_point_p2'] = 1

Validate break-point logic.

In [ ]:
bp_row = df.loc[(df["break_point_p1"] == 1) | (df["break_point_p2"] == 1)].iloc[0]

match_id = bp_row["match_id"]
set_no   = bp_row["SetNo"]
game_no  = bp_row["GameNo"]

print("Example break-point game:", match_id, "Set", set_no, "Game", game_no)
print("Row index:", bp_row.name)
print("Break point flags on that row:",
      "bp_p1=", int(bp_row["break_point_p1"]),
      "bp_p2=", int(bp_row["break_point_p2"]),
      "server=", int(bp_row["server"]),
      "receiver=", int(bp_row["receiver"]))


Example break-point game: 2011-ausopen-2501 Set 1 Game 3
Row index: 20
Break point flags on that row: bp_p1= 1 bp_p2= 0 server= 2 receiver= 1


In [18]:
game_df = (
    df[(df["match_id"] == match_id) & (df["SetNo"] == set_no) & (df["GameNo"] == game_no)]
    .sort_values("PointNumber")
)

cols = [
    "match_id","SetNo","GameNo","PointNumber",
    "PointServer","PointWinner","server","receiver",
    "p1_pts_game_before","p2_pts_game_before","game_score_str",
    "break_point_p1","break_point_p2","tiebreak"
]

# only show columns that exist (keeps it robust)
cols = [c for c in cols if c in game_df.columns]

game_df[cols]


,match_id,SetNo,GameNo,PointNumber,PointServer,PointWinner,server,receiver,p1_pts_game_before,p2_pts_game_before,game_score_str,break_point_p1,break_point_p2,tiebreak
13,2011-ausopen-2501,1,3,13,2,2,2,1,0,0,0-0,0,0,0
14,2011-ausopen-2501,1,3,14,2,2,2,1,0,1,0-15,0,0,0
15,2011-ausopen-2501,1,3,15,2,1,2,1,0,2,0-30,0,0,0
16,2011-ausopen-2501,1,3,16,2,1,2,1,1,2,15-30,0,0,0
17,2011-ausopen-2501,1,3,17,2,2,2,1,2,2,30-30,0,0,0
18,2011-ausopen-2501,1,3,18,2,1,2,1,2,3,30-40,0,0,0
19,2011-ausopen-2501,1,3,19,2,1,2,1,3,3,40–40,0,0,0
20,2011-ausopen-2501,1,3,20,2,2,2,1,4,3,Ad P1,1,0,0
21,2011-ausopen-2501,1,3,21,2,2,2,1,4,4,40–40,0,0,0
22,2011-ausopen-2501,1,3,22,2,2,2,1,4,5,Ad P2,0,0,0


Next add flags similarly for game-point, set-point, match-point

Set-points.

In [ ]:
import numpy as np

required = [
    "p1_pts_game_before", "p2_pts_game_before",
    "P1GamesWon", "P2GamesWon",
    "tiebreak",
    "p1_tb_points", "p2_tb_points"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

df["P1GamesWon"] = pd.to_numeric(df["P1GamesWon"], errors="coerce").fillna(0).astype("int64")
df["P2GamesWon"] = pd.to_numeric(df["P2GamesWon"], errors="coerce").fillna(0).astype("int64")

df["p1_pts_game_before"] = pd.to_numeric(df["p1_pts_game_before"], errors="coerce").fillna(0).astype("int64")
df["p2_pts_game_before"] = pd.to_numeric(df["p2_pts_game_before"], errors="coerce").fillna(0).astype("int64")
df["tiebreak"] = pd.to_numeric(df["tiebreak"], errors="coerce").fillna(0).astype("int64")

df["p1_tb_points"] = pd.to_numeric(df["p1_tb_points"], errors="coerce").fillna(0).astype("int64")
df["p2_tb_points"] = pd.to_numeric(df["p2_tb_points"], errors="coerce").fillna(0).astype("int64")

p1_before = df["p1_pts_game_before"]
p2_before = df["p2_pts_game_before"]

p1_after_if_win = p1_before + 1
p2_after_if_win = p2_before + 1

p1_wins_game_if_wins_point = (p1_after_if_win >= 4) & ((p1_after_if_win - p2_before) >= 2)
p2_wins_game_if_wins_point = (p2_after_if_win >= 4) & ((p2_after_if_win - p1_before) >= 2)

p1_games_new = df["P1GamesWon"] + p1_wins_game_if_wins_point.astype("int64")
p2_games_new = df["P2GamesWon"] + p2_wins_game_if_wins_point.astype("int64")

p1_set_point_in_normal_game = (
    (df["tiebreak"] == 0) &
    p1_wins_game_if_wins_point &
    (p1_games_new >= 6) &
    ((p1_games_new - df["P2GamesWon"]) >= 2)
)

p2_set_point_in_normal_game = (
    (df["tiebreak"] == 0) &
    p2_wins_game_if_wins_point &
    (p2_games_new >= 6) &
    ((p2_games_new - df["P1GamesWon"]) >= 2)
)

p1_tb_after = df["p1_tb_points"] + 1
p2_tb_after = df["p2_tb_points"] + 1

p1_wins_tb_if_wins_point = (p1_tb_after >= 7) & ((p1_tb_after - df["p2_tb_points"]) >= 2)
p2_wins_tb_if_wins_point = (p2_tb_after >= 7) & ((p2_tb_after - df["p1_tb_points"]) >= 2)

p1_set_point_in_tiebreak = (df["tiebreak"] == 1) & p1_wins_tb_if_wins_point
p2_set_point_in_tiebreak = (df["tiebreak"] == 1) & p2_wins_tb_if_wins_point

df["set_point_p1"] = (p1_set_point_in_normal_game | p1_set_point_in_tiebreak).astype("int64")
df["set_point_p2"] = (p2_set_point_in_normal_game | p2_set_point_in_tiebreak).astype("int64")

Validate.

In [ ]:
n_sets = (
    df[["match_id", "SetNo"]]
    .drop_duplicates()
    .shape[0]
)
n_set_points_p1 = df["set_point_p1"].sum()
n_set_points_p2 = df["set_point_p2"].sum()
n_set_points_total = n_set_points_p1 + n_set_points_p2

print("Total sets:", n_sets)
print("Set points (P1):", int(n_set_points_p1))
print("Set points (P2):", int(n_set_points_p2))
print("Set points (total):", int(n_set_points_total))

Total sets: 782
Set points (P1): 1077
Set points (P2): 988
Set points (total): 2065


Match-points.

In [25]:
df['match_point_p1'] = (
    (df['P1SetsWon'] == 1) &
    (df['set_point_p1'] == 1)
).astype(int)

df['match_point_p2'] = (
    (df['P2SetsWon'] == 1) &
    (df['set_point_p2'] == 1)
).astype(int)


In [27]:
# Pick one match that contains a match point and print the whole match

mp_rows = df[(df["match_point_p1"] == 1) | (df["match_point_p2"] == 1)]
assert len(mp_rows) > 0, "No match points found in df."

# take the first such match
mid = mp_rows.iloc[0]["match_id"]

match = df[df["match_id"] == mid].copy()
match = match.sort_values(["SetNo", "GameNo", "PointNumber"])

cols = [
    "match_id", "SetNo", "GameNo", "PointNumber",
    "PointServer", "PointWinner",
    "p1_pts_game_before", "p2_pts_game_before",
    "game_score_str",
    "P1GamesWon", "P2GamesWon",
    "P1SetsWon", "P2SetsWon",
    "tiebreak",
    "set_point_p1", "set_point_p2",
    "match_point_p1", "match_point_p2",
    "is_game_end", "is_set_end", "is_match_end"
]

# keep only existing columns (safe mid-pipeline)
cols = [c for c in cols if c in match.columns]

print(f"Full match with match points: {mid}")
print(match[cols].to_string(index=False))


Full match with match points: 2011-ausopen-2501
         match_id  SetNo  GameNo  PointNumber  PointServer  PointWinner  p1_pts_game_before  p2_pts_game_before game_score_str  P1GamesWon  P2GamesWon  P1SetsWon  P2SetsWon  tiebreak  set_point_p1  set_point_p2  match_point_p1  match_point_p2  is_game_end  is_set_end  is_match_end
2011-ausopen-2501      1       1            0            0            0                   0                   0            0-0           0           0          0          0         0             0             0               0               0        False       False         False
2011-ausopen-2501      1       1            1            2            2                   0                   0            0-0           0           0          0          0         0             0             0               0               0        False       False         False
2011-ausopen-2501      1       1            2            2            1                   0               

Save (almost) final dataset with all necesarry features added.

In [29]:
OUT_DIR = DATA_PROCESSED
out_path = OUT_DIR / "03_tiebreak_pressure_flags.parquet"
df.to_parquet(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df.shape)

Saved: /Users/leventezsiga/Documents/VU/Thesis_P2-P3/tennis-pressure/data_processed/03_tiebreak_pressure_flags.parquet
Shape: (48211, 51)
